# はじめに

このノートブックでは、KerasNLPライブラリを使用して、Twitterのツイートが本当に災害に関するものかどうかを分類するモデルを構築します。特に、事前にトレーニングされたDistilBERTモデル(※1)を使用してツイートのテキストを処理し、分類するモデルをトレーニングし、最終的な提出ファイルを生成していきます。

**以下のステップを順に説明していきます:**

- 災害ツイートのデータセットを読み込む
- データセットの探索
- データの前処理
- KerasNLPからDistilBERTモデルを読み込む
- モデルのトレーニング（BERTのファインチューニング）
- 提出ファイルの生成

## 必要なライブラリのインストール / TensorFlowを使用したKeras Coreの設定

In [ ]:
!pip install keras-core==0.1.7
!pip install -q keras-nlp==0.6.1

# This sample uses Keras Core, the multi-backend version of Keras.
# The selected backend is TensorFlow (other supported backends are 'jax' and 'torch')
import os
os.environ['KERAS_BACKEND'] = 'tensorflow'

## ライブラリインポート

もしこの部分でエラーが出たら、もう一度`Run`してみてください。

正しい結果は以下の通りです

```
TensorFlow version: 2.12.0
KerasNLP version: 0.6.1
```

In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import tensorflow as tf
import keras_core as keras
import keras_nlp
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

print("TensorFlow version:", tf.__version__)
print("KerasNLP version:", keras_nlp.__version__)

## データセットのロード

次に、災害ツイートのデータセットを読み込みます。`train.csv` には訓練用データ、`test.csv` にはテスト用データが含まれています。

これらのデータには、以下の情報が含まれています。

* **id**: 一意の識別子
* **keyword**: そのツイートに関連するキーワード（ただし、空白の場合もあります！）
* **location**: ツイートが送信された場所（これも空白の場合があります）
* **text**: ツイートの本文
* **target**: ツイートが実際の災害に関するものであれば `1`、そうでなければ `0`

In [ ]:
df_train = pd.read_csv("/kaggle/input/nlp-getting-started/train.csv")
df_test = pd.read_csv("/kaggle/input/nlp-getting-started/test.csv")

print('Training Set Shape = {}'.format(df_train.shape))
print('Training Set Memory Usage = {:.2f} MB'.format(df_train.memory_usage().sum() / 1024**2))
print('Test Set Shape = {}'.format(df_test.shape))
print('Test Set Memory Usage = {:.2f} MB'.format(df_test.memory_usage().sum() / 1024**2))

## データの確認

In [ ]:
df_train.head()

In [ ]:
df_test.head()

## データセットの探索

データの基本的な統計を確認し、各ツイートのテキスト長を調べていきましょう。

In [ ]:
df_train["length"] = df_train["text"].apply(lambda x : len(x))
df_test["length"] = df_test["text"].apply(lambda x : len(x))

print("Train Length Stat")
print(df_train["length"].describe())
print()

print("Test Length Stat")
print(df_test["length"].describe())

If you want to know more information about the data, you can grab useful information [here](https://www.kaggle.com/code/gunesevitan/nlp-with-disaster-tweets-eda-cleaning-and-bert)

Note that all the tweets are in english.

## データの前処理

In [ ]:
BATCH_SIZE = 32
NUM_TRAINING_EXAMPLES = df_train.shape[0]
TRAIN_SPLIT = 0.8
VAL_SPLIT = 0.2
STEPS_PER_EPOCH = int(NUM_TRAINING_EXAMPLES)*TRAIN_SPLIT // BATCH_SIZE

EPOCHS = 2
AUTO = tf.data.experimental.AUTOTUNE

訓練データをトレーニングセットとバリデーションセットに分割し、KerasNLPモデルにデータを読み込ませるように前処理を行います。

In [ ]:
from sklearn.model_selection import train_test_split

X = df_train["text"]
y = df_train["target"]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=VAL_SPLIT, random_state=42)

X_test = df_test["text"]

## DistilBERTモデルの読み込み

それでは、KerasNLPの事前学習済みモデル**DistilBERT**を使用して、テキスト分類タスクに取り組んでいきます。

BERTにテキストを入力するには、まず生テキストを数値の**トークンID**に変換し、複数の**テンソル**として整理する必要があります。

`BertClassifier` モデルは**プリプロセッサレイヤー**を設定できます。これにより、`fit()`、`predict()`、`evaluate()` の際に、生入力に対して前処理が自動的に適用されます。`from_preset()` メソッドを使ってモデルを作成する場合、この設定はデフォルトで行われます。

ここで選択した**DistilBERTモデル**は、BERTの軽量版（蒸留版）で、パフォーマンスの97%を維持しながら、パラメータ数を半分に削減しています（[論文](https://arxiv.org/abs/1910.01108)）。

`bert-base-uncased` と比べて、パラメータは40%少なく、動作速度は60%速いにもかかわらず、GLUE言語理解ベンチマークで測定されたBERTのパフォーマンスの95%以上を維持しています。

具体的には、DistilBERTにはトークンタイプ埋め込みやプーラー層がなく、GoogleのBERTの層の半分しか持っていません。

In [ ]:
# DistilBERTモデルの読み込み
preset= "distil_bert_base_en_uncased"

# Use a shorter sequence length.
preprocessor = keras_nlp.models.DistilBertPreprocessor.from_preset(preset,
                                                                   sequence_length=160,
                                                                   name="preprocessor_4_tweets"
                                                                  )

# Pretrained classifier.
classifier = keras_nlp.models.DistilBertClassifier.from_preset(preset,
                                                               preprocessor = preprocessor, 
                                                               num_classes=2)

classifier.summary()

## モデルのコンパイルとトレーニング

モデルをコンパイルし、トレーニングします。最初に学習率の低いAdamオプティマイザを使用し、損失関数として`SparseCategoricalCrossentropy`を設定します。

In [ ]:
# モデルのコンパイル
classifier.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True), #'binary_crossentropy',
    optimizer=keras.optimizers.Adam(1e-5),
    metrics= ["accuracy"]  
)

# モデルのトレーニング
history = classifier.fit(x=X_train,
                         y=y_train,
                         batch_size=BATCH_SIZE,
                         epochs=EPOCHS, 
                         validation_data=(X_val, y_val)
                         #callbacks=[early_stop], #early_stop
                         #class_weight=class_weight
                        )

## 混同行列の表示

訓練セットおよびバリデーションセットに対して予測を行い、混同行列とF1スコアを表示します。

In [ ]:
# 混同行列とF1スコアを表示する関数
def displayConfusionMatrix(y_true, y_pred, dataset):
    disp = ConfusionMatrixDisplay.from_predictions(
        y_true,
        np.argmax(y_pred, axis=1),
        display_labels=["Not Disaster","Disaster"],
        cmap=plt.cm.Blues
    )

    tn, fp, fn, tp = confusion_matrix(y_true, np.argmax(y_pred, axis=1)).ravel()
    f1_score = tp / (tp+((fn+fp)/2))

    disp.ax_.set_title("Confusion Matrix on " + dataset + " Dataset -- F1 Score: " + str(f1_score.round(2)))


### 訓練データに対して予測を行い、混同行列とF1スコアを表示

In [ ]:
y_pred_train = classifier.predict(X_train)

displayConfusionMatrix(y_train, y_pred_train, "Training")

### バリデーションデータに対して予測を行い、混同行列とF1スコアを表示

In [ ]:
y_pred_val = classifier.predict(X_val)

displayConfusionMatrix(y_val, y_pred_val, "Validation")

## 提出ファイルの生成

テストデータに対して予測を行い、提出用の`submission.csv`ファイルを作成します

In [ ]:
sample_submission = pd.read_csv("/kaggle/input/nlp-getting-started/sample_submission.csv")
# sample_submission.head()

sample_submission["target"] = np.argmax(classifier.predict(X_test), axis=1)

sample_submission.describe()

sample_submission.to_csv("submission.csv", index=False)